## Download srWGS exome VCFs

In [ ]:
%load_ext google.cloud.bigquery
# !pip install pyspark

import os
import numpy as np
import pandas as pd
from tqdm import tqdm
pd.set_option('display.max_columns', None)

# Get the BigQuery curated dataset for the current workspace context.
CDR = os.environ['WORKSPACE_CDR'] # CDR_STORAGE_PATH, HG38_REFERENCE_FASTA, WORKSPACE_BUCKET

bucket = os.getenv('WORKSPACE_BUCKET')
genomic_location = os.getenv("CDR_STORAGE_PATH")


In [ ]:
import hail as hl

hl.default_reference(new_default_reference = "GRCh38")
# manifest_path = "gs://fc-aou-datasets-controlled/v8/wgs/long_read/manifest.csv"
# manifest = pd.read_csv(manifest_path, storage_options={"requester_pays": True})
# manifest.columns

In [ ]:
exome_vcf_path = os.getenv("WGS_EXOME_VCF_PATH")
exome_vcf_path

In [ ]:
# Get total folder size
!gsutil -u $GOOGLE_PROJECT -m du -sh $WGS_EXOME_VCF_PATH/*

In [ ]:
# Get total file numbers
exome_files = !gsutil -u $$GOOGLE_PROJECT -m ls $$WGS_EXOME_VCF_PATH
len(exome_vcf)

In [ ]:
vcfs_only = !gsutil -u $GOOGLE_PROJECT -m ls $WGS_EXOME_VCF_PATH/*.vcf.bgz
len(vcfs_only)

In [ ]:
vcfs_only[:10]

In [ ]:
tbis_only = !gsutil -u $GOOGLE_PROJECT -m ls $WGS_EXOME_VCF_PATH/*.vcf.bgz.tbi
len(tbis_only)

In [ ]:
tbis_only[:10]

### Download VCF to local workspace

In [ ]:
!gsutil -u $$GOOGLE_PROJECT -m cp $WGS_EXOME_VCF_PATH/0000000000.vcf.bgz.tbi WGS_EXOME_VCF/

In [ ]:
intervals_only = !gsutil -u $GOOGLE_PROJECT -m ls $WGS_EXOME_VCF_PATH/*.interval_list
len(intervals_only)

### Step 1. Download all interval_list files locally

In [ ]:
!gsutil -u $$GOOGLE_PROJECT -m cp $WGS_EXOME_VCF_PATH/*interval_list WGS_EXOME_VCF/

### Step 2. Create a merged interval_list file

In [ ]:
import os
import glob
import io
import pandas as pd
from tqdm import tqdm

remote = False

if remote:
    interval_files = !gsutil -u $GOOGLE_PROJECT -m ls $WGS_EXOME_VCF_PATH/*.interval_list
else:
    interval_files = glob.glob(os.path.join("WGS_EXOME_VCF","*.interval_list"))
    

df = []
for f in tqdm(interval_files):
    
    os.environ['INTERVAL_FILE'] = f
    
    if remote:
        txt = !gsutil -m -u $GOOGLE_PROJECT cat $INTERVAL_FILE  | grep -v '^@'
    else:
        txt = !cat $INTERVAL_FILE | grep -v '^@'

    # Convert to pandas 
    df_tmp = pd.read_csv(io.StringIO("\n".join(txt)), 
                sep="\t", 
                header=None, 
                names=['chr','start','end','strand',''])
    df_tmp['file'] = f
    
    df.append(df_tmp)
    
df = pd.concat(df)
df['shard_id'] = df['file'].str.split(os.sep).str[-1].str.split(".").str[0]
df['vcf'] = os.getenv("WGS_EXOME_VCF_PATH")+"/"+df['shard_id']+".vcf.bgz"

df.to_csv("interval_list.all.tsv.gz",
          sep="\t")

In [ ]:
print(df.shape)
df.head()

### Step 3. Delete temporary interval_list files 

In [ ]:
!rm WGS_EXOME_VCF/*interval_list

### Step 4. Run ProHap

In [ ]:
chrom_df = df[df['chr']=='chr22']

chrom_vcfs = sorted(chrom_df['vcf'].unique().tolist())
print(len(chrom_vcfs))
chrom_vcfs[-10:]

In [ ]:
!echo $WGS_EXOME_VCF_PATH

In [ ]:
for vcf_file in tqdm(chrom_vcfs):
    vcf_local = os.path.join("WGS_EXOME_VCF", os.path.basename(vcf_file))
    if os.path.exists(vcf_local):
        continue
    os.environ['VCF_FILE'] = vcf_file
    !gsutil -u $GOOGLE_PROJECT -m cp $VCF_FILE WGS_EXOME_VCF/

### Get AoU population metadata

In [ ]:
!gsutil -u $GOOGLE_PROJECT -m cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv .

In [ ]:
ancestry = pd.read_csv("ancestry_preds.tsv", sep="\t")
ancestry.head()

In [ ]:
print(ancestry['ancestry_pred'].value_counts())
print(ancestry['ancestry_pred_other'].value_counts())

In [ ]:
ancestry['Sample name'] = ancestry['research_id'].astype(str)
ancestry['Superpopulation code'] = ancestry['ancestry_pred'].str.upper()
ancestry['Population code'] = ancestry['ancestry_pred_other'].str.upper()

ancestry_prohap = ancestry[['Sample name','Superpopulation code','Population code']]
print(ancestry_prohap.head())

ancestry_prohap.to_csv("ProHap/data/AoU_meta.tsv", sep="\t", index=False)

### Run ProHap pipeline: sharded

In [ ]:
def make_shard_yaml(shard_id,
                    input_yml=os.path.join(os.environ['PWD'],
                                            "VEP_protein/config/ProHap/All_of_Us.yaml"),
                    ouput_yml=os.path.join(os.environ['PWD'],
                                           "ProHap/config.yaml")):
    import yaml
    
    # Get yaml template
    with open(input_yml, "r") as file:
        yml_temp = yaml.safe_load(file)

    # Fill yaml template
    yml_filled = {}
    for k,v in yml_temp.items():
        if isinstance(v,str):
            v = v.replace('{shard_id}', str(shard_id))
        yml_filled[k] = v
        
    yml_filled['phased_vcf_file_name'] = yml_filled['phased_vcf_file_name'].replace(".vcf",".vcf.gz")

    # Save filled yaml
    if ouput_yml is not None:
        with open(ouput_yml, "w") as file:
            yaml.dump(yml_filled, file)
            
    # Return config as dict
    return yml_filled
        
# shard_id = chrom_df['shard_id'].tolist()[0]
# yml_filled = make_shard_yaml(shard_id, ouput_yml=None)
# yml_filled

def run_prohap(snakefile_path = os.path.join(os.environ['PWD'],
                                             "ProHap/Snakefile"),
               config=None,
               cores=None,
               use_conda=True,
               printshellcmds=True,
               **kwargs):
    import snakemake   
    
    # Select N cores
    if cores is None:
        import multiprocessing
        cores = multiprocessing.cpu_count()
        print(f"Using {cores} cores.")
    
    # Execute the Snakemake workflow
    success = snakemake.snakemake(
        snakefile=snakefile_path,
        config=config,
        cores=cores,
        use_conda=use_conda,
        printshellcmds=printshellcmds,
        **kwargs
    )
    return success


def download_vcf(chrom,
                 save_dir = os.path.join(os.environ['PWD'],"WGS_EXOME_VCF"),
                 suffix=None):
    import subprocess
    import sys
    
    vcf_local = os.path.join(save_dir, 
                             f"AoU_{chrom}_{os.path.basename(vcf_file)}")
    if suffix is not None:
        vcf_local = vcf_local.replace('.bgz',suffix)
        
    
    if not os.path.exists(vcf_local):
        os.environ['VCF_FILE'] = vcf_file
        os.environ['VCF_FILE_LOCAL'] = vcf_local

        # Run gsutil command with subprocess and stream output in real-time
        cmd = f"gsutil -u {os.environ['GOOGLE_PROJECT']} -m cp {os.environ['VCF_FILE']} {os.environ['VCF_FILE_LOCAL']}"
        process = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True
        )
        
        # Print output in real-time
        for line in process.stdout:
            print(line, end='')
            sys.stdout.flush()
            
        # Wait for process to complete and get return code
        return_code = process.wait()
        if return_code != 0:
            print(f"Error: gsutil command failed with return code {return_code}")
    
    return vcf_local

def make_snakefile(chrom,
                   input_file=os.path.join(os.environ['PWD'],"ProHap/Snakefile"),
                   output_file="auto"):
    
    if output_file=='auto':
        output_file = input_file+"_"+chrom

    with open(input_file, "r") as f:
        snakefile = f.read().split("\n")
    

    for i,val in enumerate(snakefile):
        if val.startswith("CHROMOSOMES"):
            print(val)
            snakefile[i] = f"CHROMOSOMES = ['{chrom.replace('chr','')}']"
            break
    
    if output_file is not None:    
        with open(output_file, "w") as f:
            f.write("\n".join(snakefile))
            
    return output_file

# make_snakefile(chrom)


def edit_snakefile(input_file=os.path.join(os.environ['PWD'],"ProHap/Snakefile"),
                   output_file="auto", 
                   replace_dict={}):

    if output_file=="auto":
        output_file = input_file
     
    with open(input_file, "r") as f:
        snakefile = f.read()
    
    if len(replace_dict)>0:
        for k,v in replace_dict.items():
            snakefile = snakefile.replace(k,v)

    if output_file is not None:    
        with open(output_file, "w") as f:
            f.write(snakefile)
            
    return output_file


# edit_snakefile(os.path.join(os.environ['PWD'],"ProHap/Snakefile_chr22"),
#               replace_dict={"data/vcf/phased/":"data/vcf/phased_0001234/"})

In [ ]:
import os
import pandas as pd

workspace_dir = os.environ['PWD']
os.chdir(os.path.join(workspace_dir, "ProHap"))

interval_df = pd.read_csv(os.path.join(workspace_dir, 
                                       "interval_list.all.tsv.gz"), 
                          sep="\t", 
                          index_col=0)

for chrom in tqdm(['chr22'], #interval_df['chr'].unique(),
                  desc="Iterating over chromosomes"):
    
    chrom_df = interval_df[interval_df['chr']==chrom]
    chrom_vcfs = chrom_df['vcf'].unique().tolist()
    
    for vcf_file in tqdm(chrom_vcfs,
                         desc="Iterating over VCFs"):

        # Get shard_id
        shard_id = os.path.basename(vcf_file).split(".")[0]

        # Download VCF file
        vcf_local = download_vcf(chrom=chrom, 
                                 save_dir=os.path.join(workspace_dir,"WGS_EXOME_VCF"),
                                 suffix=".gz")

        # Create yaml file
        yml_filled = make_shard_yaml(shard_id)

        
        # Make chrom-specific snakefile
        snakefile_path = make_snakefile(chrom=chrom)
        
        snakefile_path = edit_snakefile(snakefile_path,
                                        replace_dict={"data/vcf/phased/":f"data/vcf/phased_{shard_id}/"})

        # Run snakemake
        success = run_prohap(snakefile_path=snakefile_path,
                             config=yml_filled)

        break
    break

In [ ]:
success

In [ ]:
import glob

vcfs_local = glob.glob(os.path.join('WGS_EXOME_VCF','*.vcf.bgz'))
vcfs_local

## Query srWGS exome VDS

In [ ]:
interval_df['chr'].unique()

In [ ]:
vds_srwgs_path = os.getenv("WGS_VDS_PATH")
vds_srwgs_path

In [ ]:
vds = hl.vds.read_vds(vds_srwgs_path)


In [ ]:
[(x,manifest[x].nunique()) for x in manifest.columns]

In [ ]:
manifest

In [ ]:
%%bash 

git clone https://github.com/ProGenNo/ProHap.git ;
cd ProHap;